# Smoke test — NPC LoRA training pipeline

**This notebook is one-off inspection/validation scaffolding, NOT the training path.**
The training path is always the script. The real smoke run is:

```bash
python scripts/train_qwen_sft.py --config configs/smoke.yaml
```

This notebook walks the same path step by step so you can *see* each stage is
healthy: label masking, the collator, model + LoRA load, the initial-loss read,
and an overfit-to-~0 on 8 rows. Every path and hyperparameter is read from
`configs/smoke.yaml` — nothing is hardcoded. It imports the real helpers from
`scripts/train_qwen_sft.py`, so what you validate here is what the script runs.

In [ ]:
# Cell 1 — imports, load the smoke config, set the seed
import sys
from pathlib import Path

import torch
from datasets import Dataset, load_from_disk
from peft import LoraConfig, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    set_seed,
)


def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(6):
        if (p / "configs" / "smoke.yaml").exists():
            return p
        p = p.parent
    raise RuntimeError("repo root (with configs/smoke.yaml) not found from " + str(start))


REPO = find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO / "scripts"))
import train_qwen_sft as tr  # the SAME helpers the training script uses

CONFIG_PATH = REPO / "configs" / "smoke.yaml"
cfg = tr.load_config(str(CONFIG_PATH))
set_seed(cfg["seed"])

print("config     :", CONFIG_PATH)
print("base_model :", cfg["model"]["base_model"])
print("data_cache :", cfg["paths"]["data_cache"])
print("subset/steps:", cfg["training"]["subset_size"], "rows /", cfg["training"]["max_steps"], "steps")

## Step 1 — label masking is correct (no GPU)

Load the pre-tokenized cache, take one row, and decode only the tokens where
`label != -100`. That decode should be the **NPC reply** — not the system
persona or the player's line. We assert the supervised fraction is strictly
between 0 and 1, and that the very first token (the system header) is masked.

In [ ]:
# Cell 2 — Step 1: inspect masking on one row
ds_full = load_from_disk(cfg["paths"]["data_cache"])
if not isinstance(ds_full, Dataset):
    ds_full = ds_full["train"]

tokenizer = AutoTokenizer.from_pretrained(cfg["model"]["base_model"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

row = ds_full[0]
input_ids, labels = row["input_ids"], row["labels"]
supervised = [t for t, l in zip(input_ids, labels) if l != -100]
frac = len(supervised) / len(labels)

print(f"supervised fraction: {frac:.1%}  ({len(supervised)}/{len(labels)} tokens)")
print("\nSupervised decode (should read as the NPC's reply):")
print("  " + tokenizer.decode(supervised))

assert 0.0 < frac < 1.0, f"supervised fraction must be in (0,1), got {frac}"
assert labels[0] == -100, "first token (system header) must be masked"
print("\nOK: only a slice of the sequence is supervised, and it starts after the prompt.")

## Step 2 — the collator pads and aligns (no GPU)

`DataCollatorForSeq2Seq(..., label_pad_token_id=-100)` is **required**: the
default collator does not pad labels and would misalign them. Collate two rows;
all three tensors must share a shape, and the shorter row's padded tail must be
`-100` in `labels`.

In [ ]:
# Cell 3 — Step 2: collate two rows, check shapes + label padding
collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)
two = [ds_full[i] for i in range(2)]
batch = collator(two)

for k, v in batch.items():
    print(f"{k:15s} {tuple(v.shape)}")

assert batch["input_ids"].shape == batch["attention_mask"].shape == batch["labels"].shape
lens = [len(r["input_ids"]) for r in two]
shorter = 0 if lens[0] <= lens[1] else 1
pad = batch["input_ids"].shape[1] - lens[shorter]
if pad > 0:
    assert (batch["labels"][shorter, -pad:] == -100).all(), "padded label tail must be -100"
    print(f"\nOK: shapes align; shorter row padded by {pad}, its tail labels are all -100.")
else:
    print("\nOK: shapes align; the two rows are equal length (no padding to check).")

## Step 3 — load the 7B + attach LoRA (GPU)

> **Heads up: this cell loads Qwen2.5-7B and may take >1 minute.**

Load once in bf16 on GPU 0, attach LoRA, `enable_input_require_grads()` (required
for grads to reach LoRA under gradient checkpointing), and disable the KV cache.
Then one forward pass under `no_grad` for the initial-loss read. The model stays
in memory for the next cells.

In [ ]:
# Cell 4 — Step 3: model + LoRA + initial-loss read  (>1 min: loads the 7B)
model = AutoModelForCausalLM.from_pretrained(
    cfg["model"]["base_model"], torch_dtype=torch.bfloat16, device_map={"": 0},
)
l = cfg["lora"]
lora_config = LoraConfig(
    r=l["r"], lora_alpha=l["alpha"], lora_dropout=l["dropout"],
    target_modules=l["target_modules"], bias=l["bias"], task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()           # trainable % should be ~0.05%-2%
model.enable_input_require_grads()
model.config.use_cache = False

batch_dev = {k: v.to(model.device) for k, v in batch.items()}
with torch.no_grad():
    loss0 = float(model(**batch_dev).loss)
print(f"\ninitial loss = {loss0:.3f}")
print("read: 1-3 normal; near ln(vocab)~=11.6 => likely label misalignment; 0 or NaN => problem")

## Step 5 — overfit 8 rows toward 0 (GPU)

Train with a **plain `Trainer`** (via `tr.NpcTrainer`) and the same collator,
gradient checkpointing **ON** (mirrors real training). Subset size and step count
come from `configs/smoke.yaml`. A healthy pipeline drives the train loss toward
~0. If it stays flat, the prime suspect is a missing `enable_input_require_grads`
under gradient checkpointing.

In [ ]:
# Cell 5 — Step 5: overfit the first 8 rows, plot the loss, assert it nears 0
ds_smoke = tr.take_first_n(ds_full, cfg["training"]["subset_size"])
print("overfitting", len(ds_smoke), "rows for", cfg["training"]["max_steps"], "steps\n")

args = TrainingArguments(**tr.build_training_kwargs(cfg))
trainer = tr.NpcTrainer(
    model=model, args=args, train_dataset=ds_smoke,
    data_collator=collator, processing_class=tokenizer,
)
trainer.train()

losses = [e["loss"] for e in trainer.state.log_history if "loss" in e]
print("\nloss curve:", [round(x, 3) for x in losses])

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 4))
    plt.plot(range(len(losses)), losses, marker="o")
    plt.xlabel("logged step"); plt.ylabel("train loss")
    plt.title("smoke overfit (8 rows)"); plt.grid(True, alpha=0.3); plt.show()
except ImportError:
    print("(matplotlib not installed — loss list above is the curve)")

assert losses[-1] < 0.5, f"overfit did not approach 0 (final loss {losses[-1]:.3f})"
print(f"\nOK: final loss {losses[-1]:.3f} — the pipeline can drive loss toward 0.")

## Step 6 (optional) — generate from the overfit adapter

Re-enable the cache and greedily generate from the exact prompt prefix of a row
the model just trained on (everything up to its first supervised token). If the
overfit worked, the generation should echo that NPC's reply. Eyeball the voice.

In [ ]:
# Cell 6 — Step 6 (optional): generate on a seen prompt
model.config.use_cache = True
model.eval()

labels0 = ds_smoke[0]["labels"]
first_sup = next(i for i, lab in enumerate(labels0) if lab != -100)
prompt_ids = torch.tensor([ds_smoke[0]["input_ids"][:first_sup]]).to(model.device)

with torch.no_grad():
    out = model.generate(
        prompt_ids,
        max_new_tokens=cfg.get("inference", {}).get("max_new_tokens", 80),
        do_sample=False,
    )
generated = tokenizer.decode(out[0][prompt_ids.shape[1]:], skip_special_tokens=True)
truth = tokenizer.decode(
    [t for t, lab in zip(ds_smoke[0]["input_ids"], labels0) if lab != -100],
    skip_special_tokens=True,
)
print("GENERATED:\n ", generated)
print("\nGROUND TRUTH (supervised reply):\n ", truth)
model.config.use_cache = False  # restore training-safe state